# Task 3 - Part 2: Text Representation

This notebook generates two text representations from labeled sentiment data:
1. Bag-of-Words (BoW) with unigrams + bigrams
2. TF-IDF weighted GloVe document embeddings

Outputs are stored in both CSV and JSON formats.

## 1. Imports

In [28]:
from pathlib import Path
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

## 2. Configuration

In [29]:
N_RECORDS = 200
RANDOM_STATE = 42

BOW_NGRAM_RANGE = (1, 2)
GLOVE_MODEL_NAME = "glove-wiki-gigaword-100"
OUTPUT_DIR_NAME = "text_representation_outputs"

TASK3_DIR = Path(".")
INPUT_CSV_PATH = TASK3_DIR / "Cleaned_Iran_War_Sentiment_with_Sentiment_Labels.csv"
OUTPUT_DIR = TASK3_DIR / OUTPUT_DIR_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_JSON = OUTPUT_DIR / "dataset_manifest.json"
BOW_CSV = OUTPUT_DIR / "bow_features.csv"
BOW_JSON = OUTPUT_DIR / "bow_features.json"
BOW_METADATA_JSON = OUTPUT_DIR / "bow_metadata.json"
GLOVE_CSV = OUTPUT_DIR / "glove_tfidf_weighted_features.csv"
GLOVE_JSON = OUTPUT_DIR / "glove_tfidf_weighted_features.json"
GLOVE_METADATA_JSON = OUTPUT_DIR / "glove_metadata.json"



## 3. Load and Validate Data

In [30]:
if not INPUT_CSV_PATH.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_CSV_PATH}")

df_raw = pd.read_csv(INPUT_CSV_PATH)
required_columns = ["final_text", "ground_truth"]
missing_columns = [col for col in required_columns if col not in df_raw.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df_clean = df_raw.copy()
df_clean["final_text"] = df_clean["final_text"].fillna("").astype(str).str.strip()
df_clean["ground_truth"] = df_clean["ground_truth"].fillna("unknown").astype(str).str.strip()
df_clean = df_clean[df_clean["final_text"] != ""].copy()

df_work = df_clean.reset_index(drop=True)
df_work.insert(0, "row_id", np.arange(len(df_work), dtype=int))

print(f"Rows loaded: {len(df_raw)}")
print(f"Rows after cleaning/filtering: {len(df_work)}")
print("Label distribution:")
print(df_work["ground_truth"].value_counts())
df_work[["row_id", "final_text", "ground_truth"]].head()

Rows loaded: 500
Rows after cleaning/filtering: 491
Label distribution:
ground_truth
neutral     291
negative    195
positive      5
Name: count, dtype: int64


,row_id,final_text,ground_truth
0,0,belgium also say trumps war besides spain coun...,negative
1,1,massive war price tag could massive problem to...,negative
2,2,write american soldier kill innocent woman chi...,negative
3,3,verdant square radio playing note bowie hawkin...,neutral
4,4,one get away reckless stupid thing like attack...,negative


## 4. Save Dataset Manifest

In [31]:
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "input_csv": str(INPUT_CSV_PATH),
    "records_used": int(len(df_work)),
    "use_all_records": bool(True),
    "bow_ngram_range": [BOW_NGRAM_RANGE[0], BOW_NGRAM_RANGE[1]],
    "glove_model_name": GLOVE_MODEL_NAME,
    "label_distribution": {k: int(v) for k, v in df_work["ground_truth"].value_counts().to_dict().items()}
}

with MANIFEST_JSON.open("w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=True)

print(f"Manifest saved: {MANIFEST_JSON}")
manifest

Manifest saved: text_representation_outputs\dataset_manifest.json


{'created_utc': '2026-04-10T18:50:29.735908+00:00',
 'input_csv': 'Cleaned_Iran_War_Sentiment_with_Sentiment_Labels.csv',
 'records_used': 491,
 'use_all_records': True,
 'bow_ngram_range': [1, 2],
 'glove_model_name': 'glove-wiki-gigaword-100',
 'label_distribution': {'neutral': 291, 'negative': 195, 'positive': 5}}

## 5. Bag-of-Words (Unigrams + Bigrams)

In [32]:
bow_vectorizer = CountVectorizer(ngram_range=BOW_NGRAM_RANGE)
X_bow = bow_vectorizer.fit_transform(df_work["final_text"])
bow_terms = bow_vectorizer.get_feature_names_out()

feature_columns = [f"bow_f{idx}" for idx in range(len(bow_terms))]
bow_dense = X_bow.toarray().astype(np.int32)

bow_df = pd.DataFrame(bow_dense, columns=feature_columns)
bow_df.insert(0, "ground_truth", df_work["ground_truth"].values)
bow_df.insert(0, "row_id", df_work["row_id"].values)

bow_df.to_csv(BOW_CSV, index=False)

bow_records = []
for i in range(X_bow.shape[0]):
    row = X_bow.getrow(i)
    sparse_counts = {str(int(idx)): int(val) for idx, val in zip(row.indices, row.data)}
    bow_records.append({
        "row_id": int(df_work.at[i, "row_id"]),
        "ground_truth": df_work.at[i, "ground_truth"],
        "bow_sparse": sparse_counts
    })

with BOW_JSON.open("w", encoding="utf-8") as f:
    json.dump({"records": bow_records}, f, indent=2, ensure_ascii=True)

bow_density = float(X_bow.nnz / (X_bow.shape[0] * X_bow.shape[1])) if X_bow.shape[1] > 0 else 0.0
bow_term_frequencies = np.asarray(X_bow.sum(axis=0)).ravel()
top_term_indices = bow_term_frequencies.argsort()[::-1][:20]
top_terms = [
    {
        "feature_index": int(i),
        "term": str(bow_terms[i]),
        "count": int(bow_term_frequencies[i])
    }
    for i in top_term_indices
]

bow_metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "matrix_shape": [int(X_bow.shape[0]), int(X_bow.shape[1])],
    "nnz": int(X_bow.nnz),
    "density": bow_density,
    "ngram_range": [BOW_NGRAM_RANGE[0], BOW_NGRAM_RANGE[1]],
    "feature_index_to_term": {str(i): str(term) for i, term in enumerate(bow_terms)},
    "top_terms": top_terms,
    "outputs": {
        "csv": str(BOW_CSV),
        "json": str(BOW_JSON)
    }
}

with BOW_METADATA_JSON.open("w", encoding="utf-8") as f:
    json.dump(bow_metadata, f, indent=2, ensure_ascii=True)


print(f"BoW shape: {X_bow.shape}")
print(f"BoW density: {bow_density:.6f}")

BoW shape: (491, 11036)
BoW density: 0.003561


## 6. TF-IDF Weighted GloVe Embeddings

In [33]:
glove_model = api.load(GLOVE_MODEL_NAME)
embedding_dim = int(glove_model.vector_size)
print(f"Loaded GloVe model: {GLOVE_MODEL_NAME}")
print(f"Embedding dimension: {embedding_dim}")

Loaded GloVe model: glove-wiki-gigaword-100
Embedding dimension: 100


In [34]:
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 1))
X_tfidf = tfidf_vectorizer.fit_transform(df_work["final_text"])
tfidf_terms = tfidf_vectorizer.get_feature_names_out()

doc_embeddings = np.zeros((X_tfidf.shape[0], embedding_dim), dtype=np.float32)
coverage_records = []

total_terms_global = 0
total_in_vocab_global = 0
rows_with_all_oov = 0

for row_idx in range(X_tfidf.shape[0]):
    row = X_tfidf.getrow(row_idx)
    indices = row.indices
    weights = row.data

    tfidf_term_count = int(len(indices))
    total_terms_global += tfidf_term_count

    weighted_sum = np.zeros(embedding_dim, dtype=np.float32)
    in_vocab_weight_sum = 0.0
    in_vocab_term_count = 0

    for term_idx, term_weight in zip(indices, weights):
        term = tfidf_terms[term_idx]
        if term in glove_model:
            weighted_sum += glove_model[term] * np.float32(term_weight)
            in_vocab_weight_sum += float(term_weight)
            in_vocab_term_count += 1

    total_in_vocab_global += in_vocab_term_count

    if in_vocab_weight_sum > 0:
        doc_embeddings[row_idx] = weighted_sum / np.float32(in_vocab_weight_sum)
    else:
        rows_with_all_oov += 1

    coverage_ratio = float(in_vocab_term_count / tfidf_term_count) if tfidf_term_count > 0 else 0.0
    coverage_records.append({
        "row_id": int(df_work.at[row_idx, "row_id"]),
        "ground_truth": df_work.at[row_idx, "ground_truth"],
        "tfidf_terms": tfidf_term_count,
        "in_vocab_terms": int(in_vocab_term_count),
        "coverage_ratio": coverage_ratio,
        "all_oov": bool(in_vocab_term_count == 0)
    })

embedding_columns = [f"glove_{i:03d}" for i in range(embedding_dim)]
glove_df = pd.DataFrame(doc_embeddings, columns=embedding_columns)
glove_df.insert(0, "ground_truth", df_work["ground_truth"].values)
glove_df.insert(0, "row_id", df_work["row_id"].values)
glove_df.to_csv(GLOVE_CSV, index=False)

glove_records = []
for i in range(len(glove_df)):
    glove_records.append({
        "row_id": int(glove_df.at[i, "row_id"]),
        "ground_truth": glove_df.at[i, "ground_truth"],
        "embedding": [float(v) for v in doc_embeddings[i]]
    })

with GLOVE_JSON.open("w", encoding="utf-8") as f:
    json.dump({"records": glove_records}, f, indent=2, ensure_ascii=True)

overall_coverage = float(total_in_vocab_global / total_terms_global) if total_terms_global > 0 else 0.0
glove_metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "glove_model_name": GLOVE_MODEL_NAME,
    "embedding_dim": embedding_dim,
    "n_rows": int(X_tfidf.shape[0]),
    "tfidf_vocab_size": int(len(tfidf_terms)),
    "total_tfidf_terms": int(total_terms_global),
    "total_in_vocab_terms": int(total_in_vocab_global),
    "overall_coverage_ratio": overall_coverage,
    "rows_with_all_oov": int(rows_with_all_oov),
    "outputs": {
        "csv": str(GLOVE_CSV),
        "json": str(GLOVE_JSON)
    },
    "coverage_per_row": coverage_records
}

with GLOVE_METADATA_JSON.open("w", encoding="utf-8") as f:
    json.dump(glove_metadata, f, indent=2, ensure_ascii=True)


print(f"Embedding matrix shape: {doc_embeddings.shape}")
print(f"Overall term coverage ratio: {overall_coverage:.4f}")
print(f"Rows with all OOV terms: {rows_with_all_oov}")

Embedding matrix shape: (491, 100)
Overall term coverage ratio: 0.9965
Rows with all OOV terms: 0


## 7. Validation and Quick Preview

In [35]:
summary = pd.DataFrame([
    {
        "representation": "Bag-of-Words",
        "rows": int(X_bow.shape[0]),
        "features": int(X_bow.shape[1]),
        "non_zero_entries": int(X_bow.nnz)
    },
    {
        "representation": "GloVe TF-IDF weighted",
        "rows": int(doc_embeddings.shape[0]),
        "features": int(doc_embeddings.shape[1]),
        "non_zero_entries": int(np.count_nonzero(doc_embeddings))
    }
])

summary

,representation,rows,features,non_zero_entries
0,Bag-of-Words,491,11036,19295
1,GloVe TF-IDF weighted,491,100,49100


In [36]:
print("BoW preview:")
display(bow_df.iloc[:3, :12])

print("GloVe preview:")
display(glove_df.iloc[:3, :12])

BoW preview:


,row_id,ground_truth,bow_f0,bow_f1,bow_f2,bow_f3,bow_f4,bow_f5,bow_f6,bow_f7,bow_f8,bow_f9
0,0,negative,0,0,0,0,0,0,0,0,0,0
1,1,negative,0,0,0,0,0,0,0,0,0,0
2,2,negative,0,0,0,0,0,0,0,0,0,0


GloVe preview:


,row_id,ground_truth,glove_000,glove_001,glove_002,glove_003,glove_004,glove_005,glove_006,glove_007,glove_008,glove_009
0,0,negative,0.024576,0.244472,0.316841,-0.038369,0.023694,0.027261,-0.212945,0.050608,0.144100,0.020294
1,1,negative,-0.192637,0.213572,0.281260,-0.332502,0.065043,-0.071980,-0.252583,-0.115391,-0.071581,0.104360
2,2,negative,0.020964,0.368436,0.512180,-0.044032,-0.041522,-0.017420,-0.187169,-0.139715,0.186069,0.231460
